# Mediapipe

In [1]:
# Códigos basado en el ejemplo proporcionado por MediaPipe

# DETECCIÓN DE LANDMARKS FACIALES
import time
import cv2
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None  

def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    output_face_blendshapes=True,
    result_callback=on_result,
)

cap = cv2.VideoCapture(0)
with FaceLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        landmarker.detect_async(mp_image, int(time.time() * 1000))

        if latest is not None:
            h, w = frame.shape[:2]
            # Dibujamos los landmarks de la cara. Son 468 puntos por cara.
            for face in latest.face_landmarks:
                for lm in face:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 1, (0, 255, 0), -1)

        cv2.imshow("Face", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # Esc
            break

cap.release()
cv2.destroyAllWindows()

In [2]:
# DETECCIÓN DE LANDMARKS EN LAS MANOS.

import time
import cv2
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None  
def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_hands=2,
    result_callback=on_result,
)

cap = cv2.VideoCapture(0)
with HandLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        landmarker.detect_async(mp_image, int(time.time() * 1000))

        # Dibujamos los landmarks. Son 21 puntos por mano.
        if latest is not None:
            h, w = frame.shape[:2]
            for hand in latest.hand_landmarks:
                for lm in hand:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 4, (0, 255, 0), -1)

        cv2.imshow("Hands", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # Esc
            break

cap.release()
cv2.destroyAllWindows()

In [3]:
# DETECCIÓN DE POSE.

import time
import cv2
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None  
def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

# Para este modelo hay 3 pesos: el lite, el full y el heavy. Lo que sería small, medium y large.
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="pose_landmarker_full.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_poses=1,
    result_callback=on_result,
)

cap = cv2.VideoCapture(0)
with PoseLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        landmarker.detect_async(mp_image, int(time.time() * 1000))

        # Dibujamos los landmarks. Son 33 puntos-
        if latest is not None:
            h, w = frame.shape[:2]
            for pose in latest.pose_landmarks:
                for lm in pose:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 4, (0, 255, 0), -1)

        cv2.imshow("Pose", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # Esc
            break

cap.release()
cv2.destroyAllWindows()